# **Predicción del riesgo de estrés financiero en asalariados de la Comunidad de Madrid**
## **TFM - Máster de Data Science**
### *Pipeline de Modelado*
---

**Inputs:**
- `src/data/02_silver/train_silver.csv` / `test_silver.csv` - split ya realizado en Silver
- `src/data/03_gold/raw/FEATURES_NUtxt` / `FEATURES_CAT.txt` - selección del EDA

**Outputs:**
- `src/data/03_gold/` - datasets preprocesados (train/test × SEL/ALL)
- `src/data/04_modelos/` - preprocessors, modelo final, parámetros Optuna
- `src/img/` - gráficos de evaluación y explicabilidad

---

| Sección | Contenido |
|---------|-----------|
| 0 | Setup y carga |
| 1 | Preprocesado - ColumnTransformer (fit solo en train) |
| 2 | Métrica objetivo |
| 3 | Baseline con cross-validation |
| 4 | GridSearchCV sobre el mejor modelo |
| 5 | Optimización fina con Optuna |
| 6 | Evaluación final sobre test |
| 7 | Explicabilidad con DALEX |
| 8 | Exportación de artefactos |

> **Regla inamovible:** el Pipeline se ajusta (`fit`) **únicamente** sobre train.
> El test set solo se `transform`. Nunca se hace un split nuevo aquí.


---
## **0. Setup y carga**


In [ ]:
import os
os.chdir('..')
os.getcwd()

'C:\\Users\\sorim\\REPO_somm14\\EVOLVE\\tfm_project\\src'

In [41]:
import os, warnings, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
os.chdir(Path(os.getcwd()).resolve())

from utils.constants_utils import (
    PATH_TRAIN_SILVER, PATH_TEST_SILVER, PATH_FEAT_NUM, PATH_FEAT_CAT, PATH_TRAIN_GOLD,
    PATH_TEST_GOLD, PALETTE, 
    C_NEUTRAL, C0, C1, COLS_LOG1P)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
)

TARGET = 'estres_financiero_alto'
RANDOM_STATE = 42

ImportError: cannot import name 'PATH_TRAIN_GOLD' from 'utils.constants_utils' (C:\Users\sorim\REPO_somm14\EVOLVE\tfm_project\src\utils\constants_utils.py)

In [23]:
#  Carga
train = pd.read_csv(PATH_TRAIN_SILVER, low_memory=False)
test  = pd.read_csv(PATH_TEST_SILVER,  low_memory=False)

X_train_raw = train.drop(columns=[TARGET, 'motivo_aumento_ingresos', 'motivo_disminucion_ingresos'])
y_train     = train[TARGET].astype(int)
X_test_raw  = test.drop(columns=[TARGET, 'motivo_aumento_ingresos', 'motivo_disminucion_ingresos'])
y_test      = test[TARGET].astype(int)

print(f'Train: {X_train_raw.shape}  |  Test: {X_test_raw.shape}')
print(f'Clase 1 en train: {y_train.mean()*100:.1f}%  |  en test: {y_test.mean()*100:.1f}%')


Train: (2357, 64)  |  Test: (590, 64)
Clase 1 en train: 15.8%  |  en test: 15.8%


In [24]:
#  Lectura de features del EDA
def leer_txt(path):
    with open(path, encoding='utf-8') as f:
        return [x.strip() for x in f.read().split(',') if x.strip()]

FEATURES_NUM = [f for f in leer_txt(PATH_FEAT_NUM) if f in X_train_raw.columns]
FEATURES_CAT = [f for f in leer_txt(PATH_FEAT_CAT) if f in X_train_raw.columns]

print(f'Features EDA - numéricas: {len(FEATURES_NUM)}  categóricas: {len(FEATURES_CAT)}')
print(f'Numéricas: {FEATURES_NUM}')
print(f'Categóricas: {FEATURES_CAT}')


Features EDA - numéricas: 15  categóricas: 23
Numéricas: ['renta_hogar_per_capita', 'gastos_vivienda', 'ratio_carga_vivienda', 'ocupacion_isco08', 'renta_neta_salarial', 'importe_alquiler', 'num_habitaciones', 'anios_experiencia', 'meses_desempleo_ref', 'cuota_hipoteca', 'renta_no_monetaria_salarial', 'meses_desempleo_5anios', 'unidades_consumo', 'horas_semana', 'precariedad_laboral']
Categóricas: ['jornada', 'pais_nacimiento', 'hogar_riesgo_pobreza', 'tipo_vivienda', 'arope_2030', 'expectativa_ingresos_12m', 'regimen_tenencia', 'puede_proteina_2dias', 'carencia_material_social_severa', 'puede_sustituir_muebles', 'puede_vacaciones', 'nivel_estudios', 'estado_salud', 'limitacion_actividad', 'hogar_carencia_material', 'arope_2020', 'sexo', 'tipo_contrato', 'cambio_ingresos_12m', 'baja_intensidad_laboral_2020', 'nacionalidad', 'puede_calefaccion_invierno', 'tipo_hogar']


## 1. **Preprocesado - ColumnTransformer**

Se construyen **dos variantes**:
- **SEL** - solo features seleccionadas en el EDA (sin multicolinealidad)
- **ALL** - todas las features disponibles (para comparar en el baseline)

Ambas se ajustan exclusivamente sobre `X_train_raw`.

**Decisiones de preprocesado:**

| Tipo | Transformación | Justificación |
|------|---------------|---------------|
| Numéricas de renta | `log1p` -> `SimpleImputer(median)` -> `StandardScaler` | Skew confirmado en EDA.5 |
| Numéricas resto | `SimpleImputer(median)` -> `StandardScaler` | Centrado para modelos lineales |
| Categóricas nominales | `SimpleImputer(most_frequent)` -> `OHE` | Sin orden semántico |
| Categóricas ordinales | `SimpleImputer(most_frequent)` -> `OrdinalEncoder` | Preserva el orden real |

In [25]:
# Variables ordinales con su orden semántico exacto
ORDINAL_VARS = {
    'nivel_estudios':            ['Hasta primaria', 'Secundaria 1a etapa',
                                  'Post-secundaria', 'Superior universitario',
                                  'Master/Doctorado'],
    'estado_salud':              ['Muy malo', 'Malo', 'Regular', 'Bueno', 'Muy bueno'],
    'limitacion_actividad':      ['Gravemente limitado', 'Limitado (no grave)', 'No limitado'],
    'grado_urbanizacion':        ['Zona poco poblada', 'Zona media', 'Zona muy poblada'],
    'cambio_ingresos_12m':       ['Han disminuido', 'Se mantienen', 'Han aumentado'],
    'expectativa_ingresos_12m':  ['Empeorar', 'Mantenerse', 'Mejorar'],
    'carga_prestamos_no_vivienda': ['Una carga pesada', 'Una carga razonable', 'Ninguna carga'],
    'carga_asistencia_medica':   ['Una carga pesada', 'Una carga razonable',
                                  'Ninguna carga', 'No ha utilizado'],
    'carga_asistencia_dental':   ['Una carga pesada', 'Una carga razonable',
                                  'Ninguna carga', 'No ha utilizado'],
    'carga_medicamentos':        ['Una carga pesada', 'Una carga razonable',
                                  'Ninguna carga', 'No ha consumido'],
}

In [31]:
def construir_preprocessor(cols_num, cols_cat):
    '''
    Construye un ColumnTransformer con tres ramas:
    · Numéricas:  log1p (rentas) + SimpleImputer(median) + StandardScaler
    · Nominales:  SimpleImputer(most_frequent) + OHE
    · Ordinales:  SimpleImputer(most_frequent) + OrdinalEncoder con orden fijo
    '''
    cols_ord = [c for c in cols_cat if c in ORDINAL_VARS]
    cols_nom = [c for c in cols_cat if c not in ORDINAL_VARS]

    # Índices de las columnas de renta dentro del subconjunto numérico
    idx_log = [cols_num.index(c) for c in COLS_LOG1P if c in cols_num]

    def _log1p_sel(X, idx=idx_log):
        X = X.copy().astype(float)
        if idx:
            X[:, idx] = np.log1p(np.clip(X[:, idx], 0, None))
        return X

    pipe_num = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('log1p',   FunctionTransformer(_log1p_sel, validate=False, feature_names_out='one-to-one')),
        ('scaler',  StandardScaler()),
    ])

    pipe_nom = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore',
                                  sparse_output=False, drop='if_binary')),
    ])

    ord_cats = [ORDINAL_VARS[c] for c in cols_ord if c in ORDINAL_VARS]
    pipe_ord = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('enc',     OrdinalEncoder(categories=ord_cats,
                                   handle_unknown='use_encoded_value',
                                   unknown_value=-1)),
    ])

    transformers = []
    if cols_num: transformers.append(('num', pipe_num, cols_num))
    if cols_nom: transformers.append(('nom', pipe_nom, cols_nom))
    if cols_ord: transformers.append(('ord', pipe_ord, cols_ord))

    return ColumnTransformer(transformers=transformers, remainder='drop',
                             verbose_feature_names_out=True)


In [32]:
# Variante SEL: features del EDA
prep_sel = construir_preprocessor(FEATURES_NUM, FEATURES_CAT)
prep_sel.fit(X_train_raw)  # FIT SOLO SOBRE TRAIN

X_train_sel = prep_sel.transform(X_train_raw)
X_test_sel  = prep_sel.transform(X_test_raw)   # solo transform, nunca fit
feat_names_sel = prep_sel.get_feature_names_out()

# Variante ALL: todas las features 
all_num = X_train_raw.select_dtypes(include='number').columns.tolist()
all_cat = X_train_raw.select_dtypes(include='object').columns.tolist()

prep_all = construir_preprocessor(all_num, all_cat)
prep_all.fit(X_train_raw)  # FIT SOLO SOBRE TRAIN

X_train_all = prep_all.transform(X_train_raw)
X_test_all  = prep_all.transform(X_test_raw)   # solo transform, nunca fit
feat_names_all = prep_all.get_feature_names_out()

print(f'SEL → train: {X_train_sel.shape}  test: {X_test_sel.shape}')
print(f'ALL → train: {X_train_all.shape}  test: {X_test_all.shape}')

# Verificación: sin NaN en la salida
for nombre, arr in [('X_train_sel', X_train_sel), ('X_test_sel', X_test_sel),
                    ('X_train_all', X_train_all), ('X_test_all', X_test_all)]:
    n_nan = np.isnan(arr).sum()
    print(f'  NaN en {nombre}: {n_nan}  {"✓" if n_nan == 0 else "✗ REVISAR"}')


SEL → train: (2357, 69)  test: (590, 69)
ALL → train: (2357, 130)  test: (590, 130)
  NaN en X_train_sel: 0  ✓
  NaN en X_test_sel: 0  ✓
  NaN en X_train_all: 0  ✓
  NaN en X_test_all: 0  ✓


In [ ]:
# Exportar datasets preprocesados a Gold
for nombre, X, y in [
    ('train_gold_sel', X_train_sel, y_train),
    ('test_gold_sel',  X_test_sel,  y_test),
    ('train_gold_all', X_train_all, y_train),
    ('test_gold_all',  X_test_all,  y_test),
]:
    fnames = feat_names_sel if 'sel' in nombre else feat_names_all
    df = pd.DataFrame(X, columns=fnames)
    df[TARGET] = y.values
    if nombre.startswith('train'):
        df.to_csv(PATH_TRAIN_GOLD / f'{nombre}.csv', index=False)
    print(f'  {nombre}.csv  {df.shape}')


  train_gold_sel.csv  (2357, 70)
  test_gold_sel.csv  (590, 70)
  train_gold_all.csv  (2357, 131)
  test_gold_all.csv  (590, 131)


---
## M.2 Métrica objetivo

### ROC-AUC — métrica principal
Con desbalanceo 1:5, la accuracy es engañosa: un clasificador que siempre predice
"sin estrés" alcanza ~84% sin aprender nada.
**ROC-AUC** mide la capacidad de separar las dos clases a través de todos los
umbrales posibles. Es insensible al desbalanceo y permite comparar modelos
independientemente del umbral elegido.

### F1-Score clase 1 — métrica secundaria
Media armónica de precision y recall de la clase "estrés alto".
En contexto de RRHH, un **falso negativo** (empleado en riesgo no detectado)
es más costoso que un falso positivo → se usa `class_weight='balanced'` en todos
los modelos que lo soporten.

### Validación: StratifiedKFold(n_splits=5)
Preserva la proporción del target en cada fold.
Resultados reportados como **media ± desviación estándar**.


---
## M.3 Baseline con cross-validation (5-fold estratificada)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scale_pos = int((y_train == 0).sum() / (y_train == 1).sum())

MODELOS_BASELINE = {
    'LogisticRegression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=200, scale_pos_weight=scale_pos,
        random_state=RANDOM_STATE, eval_metric='logloss',
        verbosity=0, n_jobs=-1),
    'LightGBM': LGBMClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}
SCORING = {'roc_auc': 'roc_auc', 'f1': 'f1'}


In [ ]:
resultados_cv = []

for variante, X_tr in [('SEL', X_train_sel), ('ALL', X_train_all)]:
    print(f'\n── Variante {variante} ──')
    for nombre, modelo in MODELOS_BASELINE.items():
        scores = cross_validate(
            modelo, X_tr, y_train,
            cv=cv, scoring=SCORING, n_jobs=-1,
        )
        auc = scores['test_roc_auc']
        f1  = scores['test_f1']
        print(f'  {nombre:<22}  AUC: {auc.mean():.4f} ± {auc.std():.4f}'
              f'   F1: {f1.mean():.4f} ± {f1.std():.4f}')
        resultados_cv.append({
            'variante': variante, 'modelo': nombre,
            'auc_mean': auc.mean(), 'auc_std': auc.std(),
            'f1_mean':  f1.mean(),  'f1_std':  f1.std(),
        })

df_cv = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
display(df_cv.round(4))


In [ ]:
# ── Gráfico comparativo ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metrica, titulo in zip(
    axes,
    [('auc_mean', 'auc_std'), ('f1_mean', 'f1_std')],
    ['ROC-AUC (CV 5-fold)', 'F1 clase 1 (CV 5-fold)'],
):
    m_col, s_col = metrica
    for variante, color, alpha in [('SEL', C0, 0.85), ('ALL', C1, 0.65)]:
        sub = df_cv[df_cv['variante'] == variante].sort_values(m_col)
        etiquetas = [f"{r['modelo']}\n({variante})" for _, r in sub.iterrows()]
        ax.barh(etiquetas, sub[m_col], xerr=sub[s_col],
                color=color, alpha=alpha, capsize=4,
                edgecolor='white', label=variante)
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Baseline — comparativa modelos y variantes de features',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PATH_IMG / 'baseline_cv.png', bbox_inches='tight')
plt.show()

# Decisión: mejor modelo y variante
mejor_fila   = df_cv.iloc[0]
MEJOR_MODELO = mejor_fila['modelo']
MEJOR_VAR    = mejor_fila['variante']
print(f'\n→ Mejor: {MEJOR_MODELO}  (variante {MEJOR_VAR})'
      f'  AUC={mejor_fila["auc_mean"]:.4f}')


---
## M.4 GridSearchCV sobre el mejor modelo

Búsqueda en grid sobre los hiperparámetros principales del modelo seleccionado.
Sirve como punto de partida para Optuna y como referencia de robustez.


In [ ]:
from sklearn.model_selection import GridSearchCV

X_tr_gs = X_train_sel if MEJOR_VAR == 'SEL' else X_train_all

GRIDS = {
    'LogisticRegression': (
        LogisticRegression(class_weight='balanced', max_iter=1000,
                           random_state=RANDOM_STATE),
        {'C': [0.01, 0.1, 1.0, 10.0], 'solver': ['lbfgs', 'saga']},
    ),
    'RandomForest': (
        RandomForestClassifier(class_weight='balanced',
                               random_state=RANDOM_STATE, n_jobs=-1),
        {'n_estimators': [100, 300], 'max_depth': [None, 10, 20],
         'min_samples_leaf': [1, 5]},
    ),
    'XGBoost': (
        XGBClassifier(scale_pos_weight=scale_pos, random_state=RANDOM_STATE,
                      eval_metric='logloss', verbosity=0, n_jobs=-1),
        {'n_estimators': [100, 300], 'max_depth': [3, 5, 7],
         'learning_rate': [0.05, 0.1, 0.2]},
    ),
    'LightGBM': (
        LGBMClassifier(class_weight='balanced', random_state=RANDOM_STATE,
                       n_jobs=-1, verbose=-1),
        {'n_estimators': [100, 300], 'max_depth': [-1, 10],
         'learning_rate': [0.05, 0.1, 0.2], 'num_leaves': [31, 63]},
    ),
}

base_gs, param_grid = GRIDS[MEJOR_MODELO]
gs = GridSearchCV(base_gs, param_grid, cv=cv, scoring='roc_auc',
                  n_jobs=-1, verbose=1, refit=True)
gs.fit(X_tr_gs, y_train)

print(f'Mejores parámetros: {gs.best_params_}')
print(f'ROC-AUC CV (GridSearch): {gs.best_score_:.4f}')

PARAMS_GS   = gs.best_params_
SCORE_GS    = gs.best_score_


In [ ]:
# ── Curva de resultados del GridSearch ───────────────────────────────────
results_gs = pd.DataFrame(gs.cv_results_)
results_gs = results_gs.sort_values('mean_test_score', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(results_gs)), results_gs['mean_test_score'],
       yerr=results_gs['std_test_score'], color=C_NEUTRAL, alpha=0.7,
       capsize=3, edgecolor='white')
ax.axhline(SCORE_GS, color=C1, linestyle='--', lw=1.5,
           label=f'Mejor: {SCORE_GS:.4f}')
ax.set_title(f'GridSearch — {MEJOR_MODELO} — todas las combinaciones',
             fontweight='bold')
ax.set_xlabel('Combinación (ordenadas por AUC desc.)')
ax.set_ylabel('ROC-AUC CV')
ax.legend()
plt.tight_layout()
plt.savefig(PATH_IMG / 'gridsearch_resultados.png', bbox_inches='tight')
plt.show()


---
## M.5 Optimización fina con Optuna

Optuna usa el algoritmo TPE (Tree-structured Parzen Estimator) para explorar
espacios continuos de hiperparámetros de forma más eficiente que un grid.
Los parámetros del GridSearch se usan como punto de partida.

**50 trials** es suficiente para una muestra de ~2.400 observaciones.
Aumentar a 100 si el modelo es XGBoost o LightGBM y el tiempo lo permite.


In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
X_tr_opt = X_train_sel if MEJOR_VAR == 'SEL' else X_train_all

def get_objective(nombre):
    def objective(trial):
        if nombre == 'LogisticRegression':
            params = {
                'C':      trial.suggest_float('C', 1e-3, 100, log=True),
                'solver': trial.suggest_categorical('solver', ['lbfgs', 'saga']),
                'class_weight': 'balanced', 'max_iter': 1000,
                'random_state': RANDOM_STATE,
            }
            model = LogisticRegression(**params)

        elif nombre == 'RandomForest':
            params = {
                'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
                'max_depth':        trial.suggest_int('max_depth', 3, 30),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
                'max_features':     trial.suggest_categorical('max_features',
                                                              ['sqrt', 'log2']),
                'class_weight': 'balanced', 'random_state': RANDOM_STATE, 'n_jobs': -1,
            }
            model = RandomForestClassifier(**params)

        elif nombre == 'XGBoost':
            params = {
                'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
                'max_depth':       trial.suggest_int('max_depth', 3, 9),
                'learning_rate':   trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
                'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree':trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
                'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
                'scale_pos_weight': scale_pos, 'random_state': RANDOM_STATE,
                'eval_metric': 'logloss', 'verbosity': 0, 'n_jobs': -1,
            }
            model = XGBClassifier(**params)

        else:  # LightGBM
            params = {
                'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
                'max_depth':       trial.suggest_int('max_depth', 3, 20),
                'learning_rate':   trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
                'num_leaves':      trial.suggest_int('num_leaves', 20, 150),
                'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree':trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
                'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
                'class_weight': 'balanced', 'random_state': RANDOM_STATE,
                'n_jobs': -1, 'verbose': -1,
            }
            model = LGBMClassifier(**params)

        return cross_val_score(model, X_tr_opt, y_train,
                               cv=cv, scoring='roc_auc', n_jobs=-1).mean()
    return objective

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(get_objective(MEJOR_MODELO), n_trials=50, show_progress_bar=True)

PARAMS_OPTUNA = study.best_params
SCORE_OPTUNA  = study.best_value
print(f'\nMejor ROC-AUC Optuna: {SCORE_OPTUNA:.4f}')
print(f'Mejores parámetros:   {PARAMS_OPTUNA}')


In [ ]:
# ── Curva de optimización ─────────────────────────────────────────────────
valores  = [t.value for t in study.trials]
mejores  = pd.Series(valores).cummax()

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(range(1, len(valores)+1), valores,
           alpha=0.3, color=C_NEUTRAL, s=20, label='Trial')
ax.plot(range(1, len(mejores)+1), mejores,
        color=C0, lw=2, label='Mejor acumulado')
ax.axhline(SCORE_GS, color=C1, linestyle='--', lw=1.5,
           label=f'GridSearch: {SCORE_GS:.4f}')
ax.set_xlabel('Trial')
ax.set_ylabel('ROC-AUC (CV 5-fold)')
ax.set_title(f'Convergencia Optuna — {MEJOR_MODELO}', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(PATH_IMG / 'optuna_curva.png', bbox_inches='tight')
plt.show()

# Importancia de hiperparámetros según Optuna
try:
    from optuna.visualization import plot_param_importances
    fig_imp = plot_param_importances(study)
    fig_imp.write_image(str(PATH_IMG / 'optuna_param_importance.png'))
    fig_imp.show()
except Exception:
    pass


In [ ]:
# ── Entrenamiento del modelo final con los mejores parámetros ────────────
# Se entrena sobre TODO el train, no solo sobre un fold
CONSTRUCTORES = {
    'LogisticRegression': lambda p: LogisticRegression(
        **p, class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'RandomForest':       lambda p: RandomForestClassifier(
        **p, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':            lambda p: XGBClassifier(
        **p, scale_pos_weight=scale_pos, random_state=RANDOM_STATE,
        eval_metric='logloss', verbosity=0, n_jobs=-1),
    'LightGBM':           lambda p: LGBMClassifier(
        **p, class_weight='balanced', random_state=RANDOM_STATE,
        n_jobs=-1, verbose=-1),
}

modelo_final = CONSTRUCTORES[MEJOR_MODELO](PARAMS_OPTUNA)
modelo_final.fit(X_tr_opt, y_train)
print(f'Modelo final entrenado: {MEJOR_MODELO}')
print(f'Features usadas: {X_tr_opt.shape[1]}')


---
## M.6 Evaluación final sobre test


In [ ]:
from sklearn.metrics import (
    roc_auc_score, f1_score, classification_report,
    confusion_matrix, RocCurveDisplay, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score,
)

X_te = X_test_sel if MEJOR_VAR == 'SEL' else X_test_all
y_pred = modelo_final.predict(X_te)
y_prob = modelo_final.predict_proba(X_te)[:, 1]

AUC_TEST = roc_auc_score(y_test, y_prob)
F1_TEST  = f1_score(y_test, y_pred)
AP_TEST  = average_precision_score(y_test, y_prob)

print(f'ROC-AUC test:          {AUC_TEST:.4f}')
print(f'F1 clase 1 test:       {F1_TEST:.4f}')
print(f'Average Precision:     {AP_TEST:.4f}')
print()
print(classification_report(y_test, y_pred,
                             target_names=['Sin estrés', 'Estrés alto']))


In [ ]:
# ── Curva ROC + Precision-Recall + Matriz de confusión ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ROC
RocCurveDisplay.from_predictions(
    y_test, y_prob,
    name=f'{MEJOR_MODELO} (AUC={AUC_TEST:.3f})',
    ax=axes[0], color=C0,
)
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4)
axes[0].set_title('Curva ROC — test set', fontweight='bold')

# Precision-Recall
prec, rec, thr = precision_recall_curve(y_test, y_prob)
axes[1].plot(rec, prec, color=C1, lw=2)
axes[1].axhline(y_test.mean(), color='black', linestyle='--',
                alpha=0.5, label=f'Baseline ({y_test.mean():.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall (AP={AP_TEST:.3f})', fontweight='bold')
axes[1].legend(fontsize=9)

# Matriz de confusión
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred),
    display_labels=['Sin estrés', 'Estrés alto'],
).plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title('Matriz de confusión — test set', fontweight='bold')

fig.suptitle(f'Evaluación final: {MEJOR_MODELO} (variante {MEJOR_VAR})',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PATH_IMG / 'evaluacion_test.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Análisis del umbral de decisión ──────────────────────────────────────
# Por defecto sklearn usa 0.5. Con desbalanceo puede ser subóptimo.
umbrales = np.arange(0.1, 0.9, 0.02)
f1s = [f1_score(y_test, (y_prob >= u).astype(int)) for u in umbrales]
precs = [precision_recall_curve(y_test, y_prob)[0][
    np.searchsorted(np.sort(y_prob)[::-1], u)] for u in umbrales]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(umbrales, f1s, color=C0, lw=2, label='F1 clase 1')
ax.axvline(umbrales[np.argmax(f1s)], color=C1, linestyle='--',
           lw=1.5, label=f'Umbral óptimo F1: {umbrales[np.argmax(f1s)]:.2f}')
ax.axvline(0.5, color='black', linestyle=':', alpha=0.5, label='Default 0.5')
ax.set_xlabel('Umbral de decisión')
ax.set_ylabel('F1 clase 1')
ax.set_title('F1 por umbral de decisión — test set', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(PATH_IMG / 'umbral_f1.png', bbox_inches='tight')
plt.show()

UMBRAL_OPTIMO = umbrales[np.argmax(f1s)]
print(f'Umbral óptimo para F1: {UMBRAL_OPTIMO:.2f}')
print(f'F1 con umbral óptimo:  {max(f1s):.4f}')
print(f'F1 con umbral default: {f1_score(y_test, (y_prob >= 0.5).astype(int)):.4f}')


---
## M.7 Explicabilidad con DALEX

DALEX (Descriptive mAchine Learning EXplanations) es un framework de explicabilidad
agnóstico al modelo. Se trabaja siempre sobre el **conjunto de test**.

| Análisis | Qué muestra |
|----------|-------------|
| Variable importance | Qué features más impactan la predicción (permutación) |
| PDP | Cómo cambia la probabilidad predicha al variar una feature |
| Break-down | Contribución de cada feature a una predicción individual |


In [ ]:
try:
    import dalex as dx
    DALEX_OK = True
except ImportError:
    print('⚠ dalex no instalado. Ejecutar: pip install dalex kaleido')
    DALEX_OK = False


In [ ]:
if DALEX_OK:
    fnames = feat_names_sel if MEJOR_VAR == 'SEL' else feat_names_all
    X_te_df = pd.DataFrame(X_te, columns=fnames)
    X_tr_df = pd.DataFrame(X_tr_opt, columns=fnames)

    explainer = dx.Explainer(
        model=modelo_final,
        data=X_tr_df,
        y=y_train.astype(float),
        label=MEJOR_MODELO,
        verbose=False,
    )

    # ── Variable importance (permutation) ────────────────────────────────
    vi = explainer.model_parts(type='difference', N=500, random_state=RANDOM_STATE)
    vi.plot()

    top_features = (
        vi.result
        .query("variable not in ['_baseline_', '_full_model_']")
        .sort_values('dropout_loss', ascending=False)
        .head(6)['variable'].tolist()
    )
    print(f'Top 6 features por permutación: {top_features}')


In [ ]:
if DALEX_OK:
    # ── PDP para las 4 features más importantes ───────────────────────────
    pdp = explainer.model_profile(
        variables=top_features[:4], N=300, type='partial'
    )
    pdp.plot()


In [ ]:
if DALEX_OK:
    # ── Break-down para una observación de cada clase ─────────────────────
    idx_c0 = np.where(y_test.values == 0)[0][0]
    idx_c1 = np.where(y_test.values == 1)[0][0]

    for idx, etiqueta in [(idx_c0, 'Sin estrés (clase 0)'),
                          (idx_c1, 'Estrés alto (clase 1)')]:
        obs = X_te_df.iloc[[idx]]
        bd  = explainer.predict_parts(
            obs, type='break_down', random_state=RANDOM_STATE
        )
        print(f'\nBreak-down — {etiqueta}')
        bd.plot(title=f'Break-down: {etiqueta}')


In [ ]:
if DALEX_OK:
    # ── SHAP summary (si el modelo lo soporta) ────────────────────────────
    try:
        shap_vals = explainer.predict_parts(
            X_te_df.head(200), type='shap', random_state=RANDOM_STATE
        )
        shap_vals.plot()
    except Exception as e:
        print(f'SHAP no disponible para {MEJOR_MODELO}: {e}')


---
## M.8 Exportación de artefactos

Se exportan todos los objetos necesarios para el script de producción:
preprocessors (ajustados sobre train), modelo final y metadatos.


In [ ]:
# ── Preprocessors ────────────────────────────────────────────────────────
with open(PATH_MODELOS / 'preprocessor_sel.pkl', 'wb') as f:
    pickle.dump(prep_sel, f)
with open(PATH_MODELOS / 'preprocessor_all.pkl', 'wb') as f:
    pickle.dump(prep_all, f)

# ── Modelo final ─────────────────────────────────────────────────────────
with open(PATH_MODELOS / 'modelo_final.pkl', 'wb') as f:
    pickle.dump(modelo_final, f)

# ── Metadatos ─────────────────────────────────────────────────────────────
meta = {
    'modelo':          MEJOR_MODELO,
    'variante':        MEJOR_VAR,
    'umbral_decision': float(UMBRAL_OPTIMO),
    'params_optuna':   PARAMS_OPTUNA,
    'scores': {
        'baseline_auc_cv':  float(mejor_fila['auc_mean']),
        'gridsearch_auc_cv':float(SCORE_GS),
        'optuna_auc_cv':    float(SCORE_OPTUNA),
        'test_auc':         float(AUC_TEST),
        'test_f1_clase1':   float(F1_TEST),
        'test_avg_precision':float(AP_TEST),
    },
    'features_num': FEATURES_NUM_EDA,
    'features_cat': FEATURES_CAT_EDA,
}
with open(PATH_MODELOS / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('Artefactos exportados:')
for p in sorted(PATH_MODELOS.glob('*')):
    print(f'  {p}')


In [ ]:
# ── Resumen final ─────────────────────────────────────────────────────────
print('═' * 60)
print('RESUMEN DEL MODELADO')
print('═' * 60)
print(f'  Modelo final:          {MEJOR_MODELO} (variante {MEJOR_VAR})')
print(f'  Baseline AUC (CV):     {mejor_fila["auc_mean"]:.4f}')
print(f'  GridSearch AUC (CV):   {SCORE_GS:.4f}')
print(f'  Optuna AUC (CV):       {SCORE_OPTUNA:.4f}')
print(f'  AUC test final:        {AUC_TEST:.4f}')
print(f'  F1 clase 1 (test):     {F1_TEST:.4f}')
print(f'  Umbral óptimo (F1):    {UMBRAL_OPTIMO:.2f}')
print('═' * 60)
